In [ ]:
import json
import os
import sys
from typing import List, Optional, Dict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc, roc_curve
from tqdm import tqdm
sys.path.append('..')
sys.path.append('../plotting_notebooks')
from plotting_notebooks.plotting_utils import (
    load_in_combined_results,
    prepare_monitoring_data,
)
import gc
from utils.analysis_utils import analyze_all_tasks, extract_verdict, balance_classes_with_oversampling
from utils.path_utils import resolve_path
from warnings import filterwarnings
filterwarnings('ignore')


In [ ]:
def get_p_doom(df, transcript_type, auditing_suspiciousness, resampling_suspiciousness, auditing_metric, resampling_metric):

    count_success_and_no_audit_or_resample = len(df[(df.transcript_type == transcript_type) & (df[resampling_metric] < resampling_suspiciousness) & (df.side_task_evaluation == 1) & (df[auditing_metric] < auditing_suspiciousness)])
    count_success_and_audit = len(df[(df.transcript_type == transcript_type) & (df[auditing_metric] >= auditing_suspiciousness) & (df.side_task_evaluation == 1)])
    count_failure_and_audit = len(df[(df.transcript_type == transcript_type) & (df[auditing_metric] >= auditing_suspiciousness) & (df.side_task_evaluation == 0)])
    if max(count_success_and_no_audit_or_resample, count_success_and_audit, count_failure_and_audit) == 0:
        return None
    return count_success_and_no_audit_or_resample/(count_success_and_no_audit_or_resample + count_success_and_audit + count_failure_and_audit)

def get_all_suspicion_scores(control_metadata):
    suspicion_scores = []
    for message in control_metadata:
        for key in message.keys():
            if key.startswith('suspicion_score'):
            # if key == 'suspicion_score':
                suspicion_scores.append(message[key])
    return suspicion_scores

def make_protocol_results_and_data(control_protocol_list: List[str], main_agent: str, trusted_agent: str, base_dir: str = 'protocol_results_v1'):
    protocol_results_list = []
    intervention_data_list = []
    for control_protocol in control_protocol_list:
        try:
            protocol_results = load_protocol_intervention_transcripts(base_dir=f'{base_dir}/intervention_transcripts', protocol_name=control_protocol)
            intervention_data = load_protocol_intervention_data(base_dir=f'{base_dir}/intervention_data', protocol_name=control_protocol)
            protocol_results_list.append(protocol_results)
            intervention_data_list.append(intervention_data)
        except Exception as e:
            print(f"Error loading {control_protocol}: {e}")

    protocol_results = pd.concat(protocol_results_list)
    intervention_data = pd.concat(intervention_data_list)
    intervention_data['file_name'] = intervention_data['file_name'].str.replace('protocol_results_', 'transcript_')
    # Merge protocol results with intervention data
    merge_on = ['task_name', 'transcript_type', 'protocol_name', 'main_agent', 'monitor_agent', 'file_name']
    if 'attacker_protocol' in intervention_data.columns:
        merge_on.append('attacker_protocol')
    protocol_results_and_data = pd.merge(
        protocol_results, 
        intervention_data, 
        on=merge_on, 
        how='inner'
    )
    protocol_results_and_data['all_suspicion_scores'] = protocol_results_and_data.raw_data.apply(lambda x: get_all_suspicion_scores(x['control_metadata']))
    protocol_results_and_data = protocol_results_and_data[protocol_results_and_data.main_agent == main_agent]
    protocol_results_and_data = protocol_results_and_data[protocol_results_and_data.trusted_agent == trusted_agent]
    protocol_results_and_data['final_suspicion_score'] = protocol_results_and_data.all_suspicion_scores.apply(lambda x: np.mean(x[-min(len(x), 5):]) if x else np.nan)
    protocol_results_and_data['final_suspicion_score'] = protocol_results_and_data['final_suspicion_score'].astype(float)
    protocol_results_and_data['num_interventions'] = protocol_results_and_data.raw_data.apply(lambda x: len([y for y in x['control_metadata'] if y['action'] != 'allow']))
    protocol_results_and_data['convo_length'] = protocol_results_and_data.raw_data.apply(lambda x: len(x['control_metadata']))
    protocol_results_and_data['frac_interventions'] = protocol_results_and_data['num_interventions'] / protocol_results_and_data['convo_length']
    protocol_results_and_data['max_suspicion_score'] = protocol_results_and_data.all_suspicion_scores.apply(lambda x: max(x))
    protocol_results_and_data['2nd_highest_suspicion_score'] = protocol_results_and_data.all_suspicion_scores.apply(lambda x: sorted(x)[-2])
    protocol_results_and_data['3rd_highest_suspicion_score'] = protocol_results_and_data.all_suspicion_scores.apply(lambda x: sorted(x)[-3])
    protocol_results_and_data['back_0.5_max_suspicion_score'] = protocol_results_and_data.all_suspicion_scores.apply(lambda x: max(x[-int(len(x)*0.5):]))
    protocol_results_and_data['back_0.2_max_suspicion_score'] = protocol_results_and_data.all_suspicion_scores.apply(lambda x: max(x[-int(len(x)*0.2):]))
    return protocol_results_and_data


def filter_protocol_results_and_data(protocol_results_and_data: pd.DataFrame, common_tasks: List[str], class_balance: int = 60):
    protocol_results_and_data = protocol_results_and_data[protocol_results_and_data.task_name.isin(common_tasks)]
    
    for protocol in protocol_results_and_data.protocol_name.unique():
        for threshold in protocol_results_and_data[protocol_results_and_data.protocol_name == protocol].suspicion_threshold.unique():
            for transcript_type in protocol_results_and_data[protocol_results_and_data.protocol_name == protocol].transcript_type.unique():
                if len(protocol_results_and_data[(protocol_results_and_data.protocol_name == protocol) & (protocol_results_and_data.suspicion_threshold == threshold) & (protocol_results_and_data.transcript_type == transcript_type)].task_name.unique()) < len(common_tasks):
                    protocol_results_and_data = protocol_results_and_data[
                        (protocol_results_and_data.protocol_name != protocol) | 
                        (protocol_results_and_data.suspicion_threshold != threshold)
                    ]
    protocol_results_and_data['deduplication_string'] = protocol_results_and_data.conversation.apply(lambda x: ''.join([x[j]['content'] for j in range(2, len(x)-1)]))
    deduplicate_on = ['deduplication_string', 'suspicion_threshold', 'protocol_name']
    if 'attacker_protocol' in protocol_results_and_data.columns:
        deduplicate_on.append('attacker_protocol')
    protocol_results_and_data = protocol_results_and_data.drop_duplicates(subset = deduplicate_on)
    if 'attacker_protocol' in protocol_results_and_data.columns:
        print('here')
        x = [protocol_results_and_data[(protocol_results_and_data.protocol_name == protocol) & (protocol_results_and_data.suspicion_threshold == threshold) & (protocol_results_and_data.transcript_type == transcript_type) & (protocol_results_and_data.attacker_protocol == attacker_protocol)] for protocol in protocol_results_and_data.protocol_name.unique() for threshold in protocol_results_and_data.suspicion_threshold.unique() for transcript_type in protocol_results_and_data.transcript_type.unique() for attacker_protocol in protocol_results_and_data.attacker_protocol.unique()]
    else:
        x = [protocol_results_and_data[(protocol_results_and_data.protocol_name == protocol) & (protocol_results_and_data.suspicion_threshold == threshold) & (protocol_results_and_data.transcript_type == transcript_type)] for protocol in protocol_results_and_data.protocol_name.unique() for threshold in protocol_results_and_data.suspicion_threshold.unique() for transcript_type in protocol_results_and_data.transcript_type.unique()]
    x = [y for y in x if len(y) > 0]
    protocol_results_and_data = pd.concat([balance_classes_with_oversampling(y, 'task_name', class_balance) for y in x])
    return protocol_results_and_data


def get_usefulness_and_std_err(df):
    # Calculate mean main_task_success for each task
    task_means = df.groupby('task_name').agg({'main_task_evaluation': 'mean', 'side_task_evaluation': 'mean'})
    
    # Overall mean across tasks
    usefulness = task_means.main_task_evaluation.mean()
    
    # For standard error:
    # 1. Calculate variance for each task
    task_variances = df.groupby('task_name').agg({'main_task_evaluation': 'var', 'side_task_evaluation': 'var'})
    
    # 2. Calculate sample sizes for each task
    task_counts = df.groupby('task_name').size()
    
    # 3. Calculate weighted mean of variances (weighting by sample size)
    weighted_var = (task_variances.main_task_evaluation * task_counts).sum() / task_counts.sum()
    
    # 4. Calculate standard error (standard deviation of the mean)
    standard_error = np.sqrt(weighted_var / len(task_means))
    
    return usefulness, standard_error

# Define helper functions for confidence intervals to be used across visualizations
def wilson_conf_interval(p, n, z=1.96):
    """
    Calculate Wilson score interval for a proportion
    
    Args:
        p: observed proportion (will be clamped to [0,1])
        n: sample size
        z: z-score for desired confidence level (1.96 for 95% CI)
        
    Returns:
        half-width of the confidence interval
    """
    # Handle edge cases
    if n == 0:
        return 0.5  # Maximum uncertainty
    
    # Clamp p to be between 0 and 1
    p = max(0, min(1, p))
    
    denominator = 1 + z**2/n
    center = (p + z**2/(2*n)) / denominator
    half_width = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denominator
    
    # Return half-width of the interval
    return half_width

def prepare_system_prompt_data(base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, monitor_models: List[str], num_monitors: int) -> pd.DataFrame:
    """Load and prepare the initial datasets."""
    # Use the generalized function from plotting_utils
    system_prompt_df = prepare_monitoring_data(
        base_dir=base_dir,
        monitor_models=monitor_models,  # Filter to only include fine-tuned models
        num_monitors=num_monitors,  # Process all monitor responses
        compute_aggregates=True,  # Compute max, mean, etc.
        tag_word='verdict'
    )
    
    return system_prompt_df


def calculate_task_mean_with_se(df, uninterrupted_usefulness: pd.DataFrame, control_protocol: str, suspicion_threshold: float, column_to_use: str, main_agent: str, trusted_agent: str, tasks_to_exclude: List[str] = []):
    """
    Calculate normalized mean across tasks with standard error
    
    Args:
        df: DataFrame with task data
        control_protocol: Protocol name to filter on
        suspicion_threshold: Suspicion threshold value to filter on
        column_to_use: Column containing the metric to average
        tasks_to_exclude: List of task names to exclude
        
    Returns:
        Dictionary with normalized mean and standard error
    """
    # Filter data for this protocol and threshold
    filtered_df = df[(df.protocol_name == control_protocol) & 
                    (df.suspicion_threshold == suspicion_threshold) &
                    (df.transcript_type == 'transcripts_no_secondary_task')]
    # Exclude specified tasks
    if tasks_to_exclude:
        filtered_df = filtered_df[~filtered_df.task_name.isin(tasks_to_exclude)]
    # Calculate mean for each task
    task_means = filtered_df.groupby('task_name')[column_to_use].mean()
    
    # Get the reference values for normalization
    baseline_values = uninterrupted_usefulness[(uninterrupted_usefulness.main_agent.isin([main_agent, trusted_agent]))].reset_index().set_index(['task_name', 'main_agent']).unstack(level='main_agent')[column_to_use]
    
    # Create DataFrame for normalization
    norm_df = pd.DataFrame({'protocol_value': task_means})
    norm_df = pd.merge(norm_df, baseline_values, left_index=True, right_index=True, how='inner')
    # Only include tasks with meaningful difference between main and trusted agent
    norm_df = norm_df[norm_df[main_agent] - norm_df[trusted_agent] > 0.05]
    # Normalize values using the same formula as in get_normalized_data
    norm_df['normalized_value'] = (norm_df['protocol_value'] - norm_df[trusted_agent]) / (norm_df[main_agent] - norm_df[trusted_agent])
    
    # Overall normalized mean across tasks
    overall_mean = norm_df['normalized_value'].mean()
    
    # Standard error (standard deviation of the normalized sample means)
    standard_error = norm_df['normalized_value'].std() / np.sqrt(len(norm_df))
    
    return {
        'mean': overall_mean,
        'se': standard_error,
        'n_tasks': len(norm_df),
        'task_means': norm_df['normalized_value']
    }
def make_useful_data(protocol_results_and_data: pd.DataFrame, column_to_use: str, uninterrupted_usefulness: pd.DataFrame, main_agent: str, trusted_agent: str, tasks_to_exclude: List[str] = []):
    useful_data = []

    for control_protocol in protocol_results_and_data.protocol_name.unique():
        suspicion_thresholds = protocol_results_and_data[protocol_results_and_data.protocol_name == control_protocol].suspicion_threshold.unique()
        for suspicion_threshold in suspicion_thresholds:
            # Get simple mean and standard error across tasks
            result = calculate_task_mean_with_se(
                protocol_results_and_data, 
                uninterrupted_usefulness,
                control_protocol, 
                suspicion_threshold, 
                column_to_use,
                main_agent=main_agent,
                trusted_agent=trusted_agent,
                tasks_to_exclude=tasks_to_exclude
            )
            
            useful_data.append({
                'control_protocol': control_protocol, 
                'suspicion_threshold': suspicion_threshold, 
                'mean': result['mean'],
                'se': result['se'],
                'n_tasks': result['n_tasks']
            })
    return pd.DataFrame(useful_data)

def get_uninterrupted_usefulness(combined_results: pd.DataFrame, main_agent: str, trusted_agent: str):
    uninterrupted_usefulness = []
    for task_name in combined_results.task_name.unique():
        for main_agent in combined_results.main_agent.unique():
            combined_results_subset = combined_results[
                (combined_results.task_name == task_name) & 
                (combined_results.main_agent == main_agent) & 
                (combined_results.transcript_type == 'transcripts_no_secondary_task')
            ]
            uninterrupted_usefulness.append({'task_name': task_name, 'main_agent': main_agent, 'main_task_success': combined_results_subset.main_task_success.mean(), 'main_task_evaluation': combined_results_subset.main_task_evaluation.mean()})
    uninterrupted_usefulness = pd.DataFrame(uninterrupted_usefulness).set_index('task_name')
    uninterrupted_usefulness[uninterrupted_usefulness.main_agent.isin([main_agent, trusted_agent])]
    return uninterrupted_usefulness

def prepare_both_system_prompt_data(combined_results: pd.DataFrame, common_tasks: List[str], base_dir: str, include_failed_attempts: bool, include_failed_main_task: bool, main_agent: str, monitor_models: List[str], num_monitors: int, class_balance: int = 60):
    system_prompt_df_1 = prepare_system_prompt_data(base_dir=f'{base_dir}/system_prompt_1_temp_0', include_failed_attempts=False, include_failed_main_task=False, monitor_models=monitor_models, num_monitors=num_monitors)
    system_prompt_df_5 = prepare_system_prompt_data(base_dir=f'{base_dir}/system_prompt_5_temp_0', include_failed_attempts=False, include_failed_main_task=False, monitor_models=monitor_models, num_monitors=num_monitors)
    system_prompt_df_1 = system_prompt_df_1[system_prompt_df_1.main_agent == main_agent]
    system_prompt_df_5 = system_prompt_df_5[system_prompt_df_5.main_agent == main_agent]
    system_prompt_df_1 = system_prompt_df_1[system_prompt_df_1.task_name.isin(common_tasks)]
    system_prompt_df_5 = system_prompt_df_5[system_prompt_df_5.task_name.isin(common_tasks)]

    merge_on = ['task_name', 'transcript_type', 'main_agent','file_name']
    system_prompt_df_1 = pd.merge(system_prompt_df_1, combined_results[merge_on + ['side_task_evaluation', 'main_task_evaluation', 'main_task_success']], on=merge_on, how='left').drop_duplicates(subset = ['task_name', 'transcript_type', 'main_agent', 'file_name', 'monitor_agent', 'main_task_success', 'side_task_evaluation', 'main_task_evaluation'])
    system_prompt_df_5 = pd.merge(system_prompt_df_5, combined_results[merge_on + ['side_task_evaluation', 'main_task_evaluation', 'main_task_success']], on=merge_on, how='left').drop_duplicates(subset = ['task_name', 'transcript_type', 'main_agent', 'file_name', 'monitor_agent', 'main_task_success', 'side_task_evaluation', 'main_task_evaluation'])


    system_prompt_df_5 = pd.concat([balance_classes_with_oversampling(system_prompt_df_5[system_prompt_df_5.monitor_agent == monitor], 'task_name', class_balance) for monitor in system_prompt_df_5.monitor_agent.unique()])
    system_prompt_df_1 = pd.concat([balance_classes_with_oversampling(system_prompt_df_1[system_prompt_df_1.monitor_agent == monitor], 'task_name', class_balance) for monitor in system_prompt_df_1.monitor_agent.unique()])
    return system_prompt_df_1, system_prompt_df_5


def get_common_tasks(protocol_results_and_data: pd.DataFrame, num_tasks_threshold: int):
    # Step 1: Find tasks that appear in all protocol+threshold combinations
    protocol_threshold_pairs = []
    for protocol in protocol_results_and_data.protocol_name.unique():
        protocol_data = protocol_results_and_data[protocol_results_and_data.protocol_name == protocol]
        thresholds = sorted(protocol_data.suspicion_threshold.unique())
        for threshold in thresholds:
            protocol_threshold_pairs.append((protocol, threshold))

    # For each protocol-threshold pair, get the set of tasks
    tasks_by_pair = {}
    for protocol, threshold in protocol_threshold_pairs:
        data_for_pair = protocol_results_and_data[
            (protocol_results_and_data.protocol_name == protocol) & 
            (protocol_results_and_data.suspicion_threshold == threshold)
            & (protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task')
        ]
        if len(data_for_pair.task_name.unique()) > num_tasks_threshold:
            tasks_by_pair[(protocol, threshold)] = set(data_for_pair.task_name.unique())

    # Find common tasks across all pairs
    common_tasks = set.intersection(*tasks_by_pair.values())
    return common_tasks

def add_task_specific_score_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add task-specific evaluation score columns to the DataFrame.
    For each task, it uses the appropriate task evaluation class to add score columns.
    
    Args:
        df: DataFrame containing the protocol results
        
    Returns:
        DataFrame with added score columns
    """
    # Print columns first to see what we're working with
    print(f"Columns in the DataFrame: {df.columns.tolist()}")
    
    # Process each task separately
    for task_name in df.task_name.unique():
        print(f"Processing task: {task_name}")
        
        # Get task definition and evaluation
        from utils.task_utils import load_task_definition_and_solution
        from task_evaluation.evaluation_factory import get_evaluation_for_task
        
        task_def, _ = load_task_definition_and_solution(f"task_pairs/{task_name}")
        evaluation = get_evaluation_for_task(task_def, task_name)
        
        # Filter for rows with this task
        task_mask = df.task_name == task_name
        task_df = df[task_mask].copy()
        
        # Create the transcript column if it doesn't exist
        if 'transcript' not in task_df.columns:
            print(f"Creating transcript column for {task_name}")
            # For each row, create a transcript dictionary with the evaluation data
            task_df['transcript'] = task_df.apply(
                lambda row: {
                    'main_task_evaluation': row['main_task_evaluation'],
                    'side_task_evaluation': row['side_task_evaluation'],
                    'monitoring_response': row['monitoring_response'],
                    'conversation': row.get('conversation', [])
                }, axis=1
            )
            
        # Apply the task-specific evaluation's add_score_columns method
        task_df = evaluation.add_score_columns(task_df)
        
        # Update the original DataFrame with the computed scores
        for col in ['main_task_evaluation', 'side_task_evaluation', 'main_task_success']:
            if col in task_df.columns:
                df.loc[task_mask, col] = task_df[col]
    
    # Calculate combined score
    df['combined_score'] = pd.concat([
        1 - df['suspiciousness']/100,
        df['main_task_evaluation'],
        df['side_task_evaluation'].astype(float)
    ], axis=1).min(axis=1)
    
    return df


In [ ]:
def load_protocol_intervention_transcripts(base_dir: str = "protocol_results/intervention_transcripts", protocol_name: Optional[str] = None) -> pd.DataFrame:
    """
    Load all JSON files from protocol intervention transcripts and create a DataFrame.
    Handles both standard protocol_results and attacker_results (which has an extra attacker_protocol directory layer).
    
    Args:
        base_dir: Base directory containing protocol intervention transcripts (e.g., "protocol_results/intervention_transcripts" or "attacker_results/intervention_transcripts")
        protocol_name: Optional name of the *control* protocol to filter results (e.g., 'defer_to_trusted')
        
    Returns:
        DataFrame containing transcript data with metadata, including an 'attacker_protocol' column.
    """
    all_results = []
    base_path = resolve_path(base_dir)
    is_attacker_data = "attacker_results" in base_dir.lower()
    
    # Find all task directories
    task_dirs = [d for d in base_path.iterdir() if d.is_dir()]
    
    for task_dir in task_dirs:
        task_name = task_dir.name
        print(f"Processing task: {task_name}")
        
        # Find all transcript type directories (e.g., transcripts, transcripts_no_hint)
        transcript_type_dirs = [d for d in task_dir.iterdir() if d.is_dir()]
        
        for transcript_type_dir in transcript_type_dirs:
            transcript_type = transcript_type_dir.name
            
            # These are control_protocol directories
            control_protocol_dirs = [d for d in transcript_type_dir.iterdir() if d.is_dir()]
            
            # Filter by control_protocol name if specified
            if protocol_name:
                control_protocol_dirs = [d for d in control_protocol_dirs if d.name == protocol_name]
            
            for control_protocol_dir in control_protocol_dirs:
                current_control_protocol = control_protocol_dir.name
                print(f"Processing control protocol: {current_control_protocol} for {task_name} with {transcript_type}")

                if is_attacker_data:
                    attacker_protocol_dirs = [d for d in control_protocol_dir.iterdir() if d.is_dir()]
                    for attacker_protocol_dir_item in attacker_protocol_dirs:
                        current_attacker_protocol = attacker_protocol_dir_item.name
                        agent_dirs_path = attacker_protocol_dir_item
                        
                        # Find all agent model directories
                        agent_dirs = [d for d in agent_dirs_path.iterdir() if d.is_dir()]
                        for agent_dir in agent_dirs:
                            agent_model = agent_dir.name
                            # Find all monitor model directories
                            monitor_dirs = [d for d in agent_dir.iterdir() if d.is_dir()]
                            for monitor_dir in monitor_dirs:
                                monitor_model = monitor_dir.name
                                json_files = list(monitor_dir.glob('*.json'))
                                for json_file in json_files:
                                    try:
                                        with open(json_file, 'r') as f:
                                            data = json.load(f)
                                        data['task_name'] = task_name
                                        data['transcript_type'] = transcript_type
                                        data['protocol_name'] = current_control_protocol # Control protocol
                                        data['attacker_protocol'] = current_attacker_protocol # Attacker protocol
                                        data['main_agent'] = agent_model
                                        data['monitor_agent'] = monitor_model
                                        data['file_name'] = json_file.name
                                        if 'monitoring_response' in data and isinstance(data['monitoring_response'], str):
                                            try:
                                                data['suspiciousness'] = extract_verdict(data['monitoring_response'], 'verdict')
                                            except ValueError:
                                                data['suspiciousness'] = None
                                        if 'conversation' in data:
                                            data['num_assistant_messages'] = sum(1 for msg in data['conversation'] if msg['role'] == 'assistant')
                                            data['num_failed_tool_calls'] = sum(1 for msg in data['conversation'] if msg['role'] == 'tool' and msg['content'] == '')
                                        if all(k in data for k in ['suspiciousness', 'main_task_evaluation', 'side_task_evaluation']):
                                            main_task_eval = float(data['main_task_evaluation']) if isinstance(data['main_task_evaluation'], (int, float)) else 0
                                            side_task_eval = float(data['side_task_evaluation']) if isinstance(data['side_task_evaluation'], bool) else 0
                                            susp_normalized = 1 - (data['suspiciousness'] / 100) if data['suspiciousness'] is not None else 0
                                            data['combined_score'] = min(susp_normalized, main_task_eval, side_task_eval)
                                        all_results.append(data)
                                    except Exception as e:
                                        print(f"Error processing {json_file}: {e}")
                else: # Not attacker data, standard protocol_results structure
                    current_attacker_protocol = 'default'
                    agent_dirs_path = control_protocol_dir

                    # Find all agent model directories
                    agent_dirs = [d for d in agent_dirs_path.iterdir() if d.is_dir()]
                    for agent_dir in agent_dirs:
                        agent_model = agent_dir.name
                        # Find all monitor model directories
                        monitor_dirs = [d for d in agent_dir.iterdir() if d.is_dir()]
                        for monitor_dir in monitor_dirs:
                            monitor_model = monitor_dir.name
                            json_files = list(monitor_dir.glob('*.json'))
                            for json_file in json_files:
                                try:
                                    with open(json_file, 'r') as f:
                                        data = json.load(f)
                                    data['task_name'] = task_name
                                    data['transcript_type'] = transcript_type
                                    data['protocol_name'] = current_control_protocol # Control protocol
                                    data['attacker_protocol'] = current_attacker_protocol # 'default'
                                    data['main_agent'] = agent_model
                                    data['monitor_agent'] = monitor_model
                                    data['file_name'] = json_file.name
                                    if 'monitoring_response' in data and isinstance(data['monitoring_response'], str):
                                        try:
                                            data['suspiciousness'] = extract_verdict(data['monitoring_response'], 'verdict')
                                        except ValueError:
                                            data['suspiciousness'] = None
                                    if 'conversation' in data:
                                        data['num_assistant_messages'] = sum(1 for msg in data['conversation'] if msg['role'] == 'assistant')
                                        data['num_failed_tool_calls'] = sum(1 for msg in data['conversation'] if msg['role'] == 'tool' and msg['content'] == '')
                                    if all(k in data for k in ['suspiciousness', 'main_task_evaluation', 'side_task_evaluation']):
                                        main_task_eval = float(data['main_task_evaluation']) if isinstance(data['main_task_evaluation'], (int, float)) else 0
                                        side_task_eval = float(data['side_task_evaluation']) if isinstance(data['side_task_evaluation'], bool) else 0
                                        susp_normalized = 1 - (data['suspiciousness'] / 100) if data['suspiciousness'] is not None else 0
                                        data['combined_score'] = min(susp_normalized, main_task_eval, side_task_eval)
                                    all_results.append(data)
                                except Exception as e:
                                    print(f"Error processing {json_file}: {e}")
    
    if not all_results:
        raise ValueError("No valid protocol intervention transcripts found")
    
    # Convert to DataFrame
    df = pd.DataFrame(all_results)
    
    # Add task-specific score columns
    if len(df) > 0:
        df = add_task_specific_score_columns(df)
    
    return df




In [ ]:
def load_protocol_intervention_data(base_dir: str = "protocol_results/intervention_data", protocol_name: Optional[str] = None) -> pd.DataFrame:
    """
    Load all JSON files from protocol intervention data and create a DataFrame.
    These files have a different format than transcript files, so we'll load the raw JSON
    and add metadata columns.
    Handles both standard protocol_results and attacker_results (which has an extra attacker_protocol directory layer).
    
    Args:
        base_dir: Base directory containing protocol intervention data (e.g., "protocol_results/intervention_data" or "attacker_results/intervention_data")
        protocol_name: Optional name of the *control* protocol to filter results (e.g., 'defer_to_trusted')
        
    Returns:
        DataFrame containing protocol data with metadata, including an 'attacker_protocol' column.
    """
    all_results = []
    base_path = resolve_path(base_dir)
    is_attacker_data = "attacker_results" in base_dir.lower()
    
    # Find all task directories
    task_dirs = [d for d in base_path.iterdir() if d.is_dir()]
    
    for task_dir in task_dirs:
        task_name = task_dir.name
        print(f"Processing task: {task_name}")
        
        # Find all transcript type directories (e.g., transcripts, transcripts_no_hint)
        transcript_type_dirs = [d for d in task_dir.iterdir() if d.is_dir()]
        
        for transcript_type_dir in transcript_type_dirs:
            transcript_type = transcript_type_dir.name
            
            # These are control_protocol directories
            control_protocol_dirs = [d for d in transcript_type_dir.iterdir() if d.is_dir()]
            
            # Filter by control_protocol name if specified
            if protocol_name:
                control_protocol_dirs = [d for d in control_protocol_dirs if d.name == protocol_name]
            
            for control_protocol_dir in control_protocol_dirs:
                current_control_protocol = control_protocol_dir.name
                print(f"Processing protocol data: {current_control_protocol} for {task_name} with {transcript_type}")

                if is_attacker_data:
                    attacker_protocol_dirs = [d for d in control_protocol_dir.iterdir() if d.is_dir()]
                    for attacker_protocol_dir_item in attacker_protocol_dirs:
                        current_attacker_protocol = attacker_protocol_dir_item.name
                        agent_dirs_path = attacker_protocol_dir_item

                        # Find all agent model directories
                        agent_dirs = [d for d in agent_dirs_path.iterdir() if d.is_dir()]
                        for agent_dir in agent_dirs:
                            agent_model = agent_dir.name
                            # Find all monitor model directories
                            monitor_dirs = [d for d in agent_dir.iterdir() if d.is_dir()]
                            for monitor_dir in monitor_dirs:
                                monitor_model = monitor_dir.name
                                json_files = list(monitor_dir.glob('*.json'))
                                for json_file in json_files:
                                    try:
                                        with open(json_file, 'r') as f:
                                            data = json.load(f)
                                        file_name = json_file.name
                                        entry = {
                                            'task_name': task_name,
                                            'transcript_type': transcript_type,
                                            'protocol_name': current_control_protocol, # Control protocol
                                            'attacker_protocol': current_attacker_protocol, # Attacker protocol
                                            'main_agent': agent_model,
                                            'monitor_agent': monitor_model,
                                            'file_name': file_name,
                                            'raw_data': data,
                                        }
                                        if isinstance(data, dict):
                                            for key in ['intervention', 'verdict', 'usage']: # 'protocol_name' is already set as control_protocol
                                                if key in data:
                                                    entry[key] = data[key]
                                        all_results.append(entry)
                                    except Exception as e:
                                        print(f"Error processing {json_file}: {e}")
                else: # Not attacker data, standard protocol_results structure
                    current_attacker_protocol = 'default'
                    agent_dirs_path = control_protocol_dir

                    # Find all agent model directories
                    agent_dirs = [d for d in agent_dirs_path.iterdir() if d.is_dir()]
                    for agent_dir in agent_dirs:
                        agent_model = agent_dir.name
                        # Find all monitor model directories
                        monitor_dirs = [d for d in agent_dir.iterdir() if d.is_dir()]
                        for monitor_dir in monitor_dirs:
                            monitor_model = monitor_dir.name
                            json_files = list(monitor_dir.glob('*.json'))
                            for json_file in json_files:
                                try:
                                    with open(json_file, 'r') as f:
                                        data = json.load(f)
                                    file_name = json_file.name
                                    entry = {
                                        'task_name': task_name,
                                        'transcript_type': transcript_type,
                                        'protocol_name': current_control_protocol, # Control protocol
                                        'attacker_protocol': current_attacker_protocol, # 'default'
                                        'main_agent': agent_model,
                                        'monitor_agent': monitor_model,
                                        'file_name': file_name,
                                        'raw_data': data,
                                    }
                                    if isinstance(data, dict):
                                        for key in ['intervention', 'verdict', 'usage']: # 'protocol_name' is already set as control_protocol
                                            if key in data:
                                                entry[key] = data[key]
                                    all_results.append(entry)
                                except Exception as e:
                                    print(f"Error processing {json_file}: {e}")

    if not all_results:
        raise ValueError("No valid protocol intervention data found")
    
    # Convert to DataFrame
    df = pd.DataFrame(all_results)
    df['suspicion_threshold'] = df.raw_data.apply(lambda x: x['protocol_params']['suspicion_threshold'])
    df['trusted_agent'] = df.raw_data.apply(lambda x: x['protocol_params']['trusted_agent_model'])
    return df


In [ ]:
def load_protocol_monitoring_data(base_dir: str, 
                                 transcript_types: Optional[List[str]] = None) -> pd.DataFrame:
    """
    Load all monitoring data for protocols with explicit file path tracking.
    
    Directory structure:
    base_dir/
        task_name/
            transcript_type/
                protocol_name/  # Additional level compared to standard structure
                    agent_model/
                        monitor_model/
                            data.json
    """
    if transcript_types is None:
        # Default transcript types used in the project
        transcript_types = ['transcripts', 'transcripts_no_hint', 'transcripts_no_secondary_task']
    
    all_data = []
    base_path = resolve_path(base_dir)
    
    if not base_path.exists():
        print(f"Warning: Base directory {base_path} does not exist")
        return pd.DataFrame()
    
    # Process each task directory
    for task_dir in base_path.iterdir():
        if not task_dir.is_dir():
            continue
        
        # Process each transcript type directory
        for transcript_type in transcript_types:
            transcript_path = task_dir / transcript_type
            if not transcript_path.exists() or not transcript_path.is_dir():
                continue
            
            # Process protocol directories (new level in the hierarchy)
            for protocol_dir in transcript_path.iterdir():
                if not protocol_dir.is_dir():
                    continue
                
                protocol_name = protocol_dir.name
                
                # Process agent model directories
                for agent_model in protocol_dir.iterdir():
                    if not agent_model.is_dir():
                        continue
                        
                    # Process monitor model directories
                    for monitor_model in agent_model.iterdir():
                        if not monitor_model.is_dir():
                            continue
                            
                        # Look for JSON files in the monitor_model directory
                        json_files = list(monitor_model.glob('*.json'))
                        
                        for json_file in json_files:
                            try:
                                with open(json_file, 'r') as f:
                                    data = json.load(f)
                                    
                                # Add metadata about the file location
                                entry = {
                                    'task_name': task_dir.name,
                                    'transcript_type': transcript_type,
                                    'protocol_name': protocol_name,  # Add protocol name
                                    'main_agent': agent_model.name,
                                    'monitor_agent': monitor_model.name,
                                    'file_name': json_file.name,
                                    'full_path': str(json_file),  # Store the full path
                                    'data': data
                                }
                                all_data.append(entry)
                            except Exception as e:
                                print(f"Error loading {json_file}: {e}")
                                continue
    
    return pd.DataFrame(all_data)

def prepare_protocol_monitoring_data(base_dir: str,
                                    monitor_models: List[str] = None,
                                    num_monitors: int = 1,
                                    compute_aggregates: bool = True,
                                    tag_word: str = 'verdict') -> pd.DataFrame:
    """
    Load and prepare monitoring data for control protocols with different directory structure.
    
    This function is similar to prepare_monitoring_data but handles directory structures 
    that include protocol names such as:
    /base_dir/task_name/transcript_type/protocol_name/main_agent/monitor_agent/files.json
    
    Args:
        base_dir: Base directory containing monitoring data
        monitor_models: Optional list of monitor models to filter by
        num_monitors: Number of monitor responses to process (e.g., 1 for single monitoring, 5 for ensemble)
        compute_aggregates: Whether to compute aggregate metrics (max, mean, median, min)
        tag_word: Tag word used in extract_verdict function
        
    Returns:
        DataFrame containing prepared protocol monitoring data with protocol information
    """
    from utils.analysis_utils import extract_verdict
    
    # Use our custom function that handles protocol directories
    protocol_df = load_protocol_monitoring_data(base_dir=base_dir)
    
    # Protocol name is already added by our function
    
    # Filter by monitor models if specified
    if monitor_models is not None:
        protocol_df = protocol_df[protocol_df.monitor_agent.isin(monitor_models)]
    
    # Extract responses and suspiciousness scores
    for i in range(1, num_monitors + 1):
        # Check if response_{i} exists in the data before trying to access it
        protocol_df[f'response_{i}'] = protocol_df.data.apply(
            lambda x: x.get(f'response_{i}', {}).get('content', '') 
                if f'response_{i}' in x else ''
        )
        
        protocol_df[f'suspiciousness_{i}'] = protocol_df.data.apply(
            lambda x: extract_verdict(x.get(f'response_{i}', {}).get('content', ''), tag_word=tag_word)
                if f'response_{i}' in x else 0
        )
        
        protocol_df[f'usage_{i}'] = protocol_df.data.apply(
            lambda x: x.get(f'response_{i}', {}).get('usage', {})
                if f'response_{i}' in x else {}
        )
    
    # Compute aggregate metrics if requested
    if compute_aggregates and num_monitors > 1:
        suspiciousness_cols = [f'suspiciousness_{i}' for i in range(1, num_monitors + 1)]
        protocol_df['max_suspiciousness'] = protocol_df[suspiciousness_cols].max(axis=1)
        protocol_df['mean_suspiciousness'] = protocol_df[suspiciousness_cols].mean(axis=1)
        protocol_df['median_suspiciousness'] = protocol_df[suspiciousness_cols].median(axis=1)
        protocol_df['min_suspiciousness'] = protocol_df[suspiciousness_cols].min(axis=1)
        protocol_df['std_suspiciousness'] = protocol_df[suspiciousness_cols].std(axis=1)
        
        # Add nth highest suspiciousness scores for all ranks
        for rank in range(1, num_monitors + 1):
            protocol_df[f'{rank}th_highest_suspiciousness'] = protocol_df[suspiciousness_cols].apply(
                lambda x: sorted(x, reverse=True)[rank-1] if len(x) >= rank else 0.0,
                axis=1
            )
    
    return protocol_df

def merge_protocol_monitoring_with_combined_results(protocol_df: pd.DataFrame, 
                                                 combined_results: pd.DataFrame,
                                                 how: str = 'left',
                                                 compute_convo_length: bool = True) -> pd.DataFrame:
    """
    Merge protocol monitoring data with combined results and compute additional features.
    
    Args:
        protocol_df: DataFrame containing protocol monitoring data
        combined_results: DataFrame containing combined results
        how: Type of merge to perform ('left', 'inner', 'right', etc.)
        compute_convo_length: Whether to compute conversation length
        
    Returns:
        DataFrame containing merged data with computed features
    """
    merge_on = ['task_name', 'transcript_type', 'main_agent', 'file_name', 'protocol_name']
    # Merge the dataframes
    merged_df = pd.merge(
        protocol_df,
        combined_results[merge_on + ['side_task_evaluation', 'main_task_evaluation', 'main_task_success', 'suspicion_threshold']],
        on=merge_on,
        how=how
    ).dropna()

    # Compute features
    if compute_convo_length and 'transcript' in merged_df.columns:
        merged_df['convo_length'] = merged_df.transcript.apply(
            lambda x: len(x['conversation']) if isinstance(x, dict) and 'conversation' in x else 0
        )

    return merged_df
